In [1]:
%pip -q install fastapi uvicorn python-multipart

from fastapi import FastAPI, Form
from fastapi.responses import HTMLResponse, RedirectResponse
from IPython.display import display, HTML
import sqlite3, threading, time, html

app = FastAPI()
DB = "rewear.db"

# Database
with sqlite3.connect(DB) as con:
    con.execute("""CREATE TABLE IF NOT EXISTS clothes(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT, size TEXT, price REAL, condition TEXT, seller TEXT
    )""")
    if con.execute("SELECT COUNT(*) FROM clothes").fetchone()[0] == 0:
        con.executemany(
            "INSERT INTO clothes(name,size,price,condition,seller) VALUES(?,?,?,?,?)",
            [
                ("Blue Shirt","M",500,"Good","ReWear"),
                ("Pretty Dress","S",900,"Like New","ReWear"),
                ("Denim Jacket","L",1200,"Good","ReWear")
            ]
        )
    con.execute("""CREATE TABLE IF NOT EXISTS orders(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        item_id INTEGER, buyer TEXT, phone TEXT, address TEXT
    )""")

def page(content):
    return HTMLResponse(f"""
    <html><head><meta name="viewport" content="width=device-width,initial-scale=1">
    <title>ReWear</title><style>
    body{{margin:0;background:#f6f5ee;font-family:Arial;color:#263d30}}
    header{{background:white;padding:20px 7%;display:flex;justify-content:space-between;flex-wrap:wrap}}
    header a{{color:#286044;text-decoration:none;font-weight:bold;margin-right:15px}}
    .hero{{padding:45px 7%;background:#e0ebdc}}
    h1{{font-size:42px}} main{{padding:30px 7%}}
    .grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:18px}}
    .card,.form{{background:white;padding:20px;border-radius:12px}}
    .price{{font-size:23px;color:#286044;font-weight:bold}}
    .btn{{background:#286044;color:white;padding:12px 17px;border:0;border-radius:6px;text-decoration:none;display:inline-block;cursor:pointer}}
    input,select,textarea{{width:100%;padding:12px;margin:7px 0 15px;box-sizing:border-box;border:1px solid #ccc;border-radius:6px}}
    label{{font-weight:bold}} .form{{max-width:550px;margin:auto}}
    </style></head><body>
    <header><a href="/" style="font-size:25px">♻ ReWear</a>
    <nav><a href="/">Home</a><a href="/sell">Sell Clothes</a></nav></header>
    {content}
    </body></html>""")

@app.get("/", response_class=HTMLResponse)
def home(q: str = ""):
    with sqlite3.connect(DB) as con:
        rows = con.execute("SELECT * FROM clothes ORDER BY id DESC").fetchall()

    cards = ""
    for item in rows:
        if q.lower() not in item[1].lower():
            continue
        cards += f"""
        <div class="card">
        <h2>{html.escape(item[1])}</h2>
        <p>Size: {html.escape(item[2])}</p>
        <p>Condition: {html.escape(item[4])}</p>
        <p class="price">Rs. {item[3]:,.0f}</p>
        <p>Seller: {html.escape(item[5])}</p>
        <a class="btn" href="/buy/{item[0]}">Buy Clothes</a>
        </div>"""

    if not cards:
        cards = "<p>No clothes found. Try another search.</p>"

    return page(f"""
    <section class="hero">
    <h1>Wear it. Love it. ReWear it.</h1>
    <p>Sell clothes you no longer wear. Buy pre-loved clothes at affordable prices.</p>
    <a class="btn" href="/sell">Sell Your Clothes</a>
    </section>
    <main><h2>Available Clothes</h2>
    <form action="/" method="get">
    <input name="q" placeholder="Search clothes..." value="{html.escape(q, quote=True)}">
    <button class="btn">Search</button></form>
    <div class="grid">{cards}</div></main>""")

@app.get("/sell", response_class=HTMLResponse)
def sell():
    return page("""
    <main><div class="form"><h1>Sell Your Clothes</h1>
    <form action="/sell" method="post">
    <label>Clothes name</label><input name="name" required>
    <label>Size</label><select name="size">
    <option>XS</option><option>S</option><option>M</option>
    <option>L</option><option>XL</option><option>One Size</option></select>
    <label>Price in rupees</label><input name="price" type="number" min="1" required>
    <label>Condition</label><select name="condition">
    <option>Like New</option><option>Good</option><option>Fair</option></select>
    <label>Your name</label><input name="seller" required>
    <button class="btn">Publish Clothes</button>
    </form></div></main>""")

@app.post("/sell")
def add(name: str = Form(...), size: str = Form(...),
        price: float = Form(...), condition: str = Form(...),
        seller: str = Form(...)):
    if price <= 0:
        return HTMLResponse("Price must be greater than zero.", status_code=400)
    with sqlite3.connect(DB) as con:
        con.execute(
            "INSERT INTO clothes(name,size,price,condition,seller) VALUES(?,?,?,?,?)",
            (name.strip(), size, price, condition, seller.strip())
        )
    return RedirectResponse("/", status_code=303)

@app.get("/buy/{item_id}", response_class=HTMLResponse)
def buy(item_id: int):
    with sqlite3.connect(DB) as con:
        item = con.execute(
            "SELECT id,name,price FROM clothes WHERE id=?", (item_id,)
        ).fetchone()
    if not item:
        return HTMLResponse("Clothes not found.", status_code=404)

    return page(f"""
    <main><div class="form"><h1>Buy Clothes</h1>
    <h3>{html.escape(item[1])} — Rs. {item[2]:,.0f}</h3>
    <form action="/buy/{item_id}" method="post">
    <label>Your name</label><input name="buyer" required>
    <label>Phone number</label><input name="phone" required>
    <label>Delivery address</label><textarea name="address" required></textarea>
    <button class="btn">Place Order Request</button>
    </form></div></main>""")

@app.post("/buy/{item_id}")
def order(item_id: int, buyer: str = Form(...),
          phone: str = Form(...), address: str = Form(...)):
    with sqlite3.connect(DB) as con:
        exists = con.execute(
            "SELECT id FROM clothes WHERE id=?", (item_id,)
        ).fetchone()
        if not exists:
            return HTMLResponse("Clothes not found.", status_code=404)
        con.execute(
            "INSERT INTO orders(item_id,buyer,phone,address) VALUES(?,?,?,?)",
            (item_id, buyer.strip(), phone.strip(), address.strip())
        )
    return RedirectResponse("/success", status_code=303)

@app.get("/success", response_class=HTMLResponse)
def success():
    return page("""
    <main class="form"><h1>Order Request Sent!</h1>
    <p>Your order request has been saved.</p>
    <a class="btn" href="/">Back to Home</a></main>""")

# Start website
if not any(t.name == "rewear_server" for t in threading.enumerate()):
    def run_server():
        import uvicorn
        uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

    threading.Thread(target=run_server, daemon=True,
                     name="rewear_server").start()
    time.sleep(2)

display(HTML("""
<h2 style="color:#286044">ReWear Website Ready!</h2>
<p>Click here to open your website:</p>
<a href="http://127.0.0.1:8000" target="_blank">
http://127.0.0.1:8000</a>
"""))


Note: you may need to restart the kernel to use updated packages.


ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8000): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted


In [3]:
@app.get("/buy/{item_id}", response_class=HTMLResponse)
def buy(item_id: int):
    with sqlite3.connect(DB) as con:
        item = con.execute(
            "SELECT id,name,price FROM clothes WHERE id=?",
            (item_id,)
        ).fetchone()

    if not item:
        return HTMLResponse("Clothes not found.", status_code=404)

    price = item[2]
    commission_rate = 0.12
    commission = round(price * commission_rate, 2)
    seller_earning = round(price - commission, 2)

    return page(f"""
    <main>
    <div class="form">
    <h1>Buy Clothes</h1>
    <h3>{html.escape(item[1])}</h3>

    <p>Clothes Price: Rs. {price:,.2f}</p>
    <p>ReWear Commission (12%): Rs. {commission:,.2f}</p>
    <p>Seller Earning: Rs. {seller_earning:,.2f}</p>

    <hr>
    <form action="/buy/{item_id}" method="post">
    <label>Your Name</label>
    <input name="buyer" required>

    <label>Phone Number</label>
    <input name="phone" required>

    <label>Delivery Address</label>
    <textarea name="address" required></textarea>

    <button class="btn">Place Order Request</button>
    </form>
    </div>
    </main>
    """)


In [4]:

price = 2000
commission_rate = 0.12

commission = round(price * commission_rate, 2)
seller_earning = round(price - commission, 2)

print("Kapron ki qeemat: Rs.", price)
print("ReWear commission (12%): Rs.", commission)
print("Seller ko milega: Rs.", seller_earning)


Kapron ki qeemat: Rs. 2000
ReWear commission (12%): Rs. 240.0
Seller ko milega: Rs. 1760.0


In [5]:

COMMISSION_RATE = 0.12

commission = round(price * COMMISSION_RATE, 2)
seller_earning = round(price - commission, 2)


In [6]:

price = 2000
commission_rate = 0.12

commission = price * commission_rate
seller_earning = price - commission

print("ReWear Commission: Rs.", commission)
print("Seller ko milega: Rs.", seller_earning)
print("Total Price: Rs.", price)


ReWear Commission: Rs. 240.0
Seller ko milega: Rs. 1760.0
Total Price: Rs. 2000


In [1]:

%pip install fastapi uvicorn


Note: you may need to restart the kernel to use updated packages.


In [2]:

%%writefile main.py
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from html import escape

app = FastAPI(title="ReWear Marketplace")

products = [
    {"id": 1, "name": "Denim Jacket", "price": 2000,
     "size": "M", "condition": "Very Good", "emoji": "🧥"},
    {"id": 2, "name": "Floral Dress", "price": 1500,
     "size": "S", "condition": "Excellent", "emoji": "👗"},
    {"id": 3, "name": "Casual Shirt", "price": 900,
     "size": "L", "condition": "Good", "emoji": "👚"},
]

orders = []
commission_rate = 0.12


class NewProduct(BaseModel):
    name: str
    price: float
    size: str = "M"
    condition: str = "Good"
    emoji: str = "👕"


class NewOrder(BaseModel):
    product_id: int


@app.get("/", response_class=HTMLResponse)
def home():
    cards = ""
    for p in products:
        cards += f"""
        <div class="card">
          <div class="photo">{escape(p['emoji'])}</div>
          <h3>{escape(p['name'])}</h3>
          <p>Size: {escape(p['size'])} ·
             {escape(p['condition'])}</p>
          <h3>Rs. {p['price']:,.0f}</h3>
          <button onclick="buy({p['id']})">
            Buy Now
          </button>
        </div>
        """

    return f"""
    <!DOCTYPE html>
    <html>
    <head>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>ReWear | Pre-loved Fashion</title>
    <style>
      * {{ box-sizing: border-box; }}
      body {{ margin:0; font-family:Arial,sans-serif;
        background:#f8f7f2; color:#243b32; }}
      header {{ background:#214d3b; color:white;
        padding:24px 7%; }}
      header h1 {{ margin:0; font-size:34px; }}
      nav {{ margin-top:12px; }}
      nav a {{ color:white; margin-right:18px;
        text-decoration:none; }}
      .hero {{ padding:55px 7%; background:#e2ecdf; }}
      .hero h2 {{ font-size:36px; margin-bottom:10px; }}
      .hero p {{ font-size:17px; }}
      section {{ padding:28px 7%; }}
      .products {{ display:grid;
        grid-template-columns:repeat(auto-fit,minmax(210px,1fr));
        gap:20px; }}
      .card {{ background:white; border-radius:16px;
        padding:18px; box-shadow:0 3px 14px #0001; }}
      .photo {{ height:140px; display:grid;
        place-items:center; background:#f0eee6;
        border-radius:12px; font-size:65px; }}
      button {{ background:#285840; color:white;
        border:0; padding:12px 16px; border-radius:8px;
        cursor:pointer; }}
      input, select {{ padding:11px; margin:6px 0;
        width:100%; border:1px solid #ccd5cb;
        border-radius:7px; }}
      .panel {{ max-width:550px; background:white;
        padding:22px; border-radius:14px; margin:12px 0; }}
      #message {{ font-weight:bold; margin-top:12px; }}
      footer {{ background:#214d3b; color:white;
        padding:24px; text-align:center; }}
    </style>
    </head>
    <body>
    <header>
      <h1>♻ ReWear</h1>
      <p>Give clothes a second life.</p>
      <nav>
        <a href="#shop">Shop</a>
        <a href="#sell">Sell Clothes</a>
        <a href="#commission">Commission</a>
      </nav>
    </header>

    <div class="hero">
      <h2>Pre-loved fashion, new beginnings.</h2>
      <p>Discover affordable fashion and make sustainable choices.</p>
      <button onclick="location.href='#shop'">Explore Clothes</button>
    </div>

    <section id="shop">
      <h2>Discover Pre-loved Clothes</h2>
      <input id="search" placeholder="Search clothes..."
        oninput="filterProducts()">
      <div class="products" id="products">{cards}</div>
    </section>

    <section id="sell">
      <h2>Sell Your Clothes</h2>
      <div class="panel">
        <form id="sellForm">
          <label>Clothing name</label>
          <input name="name" required maxlength="80"
            placeholder="e.g. Blue Denim Jacket">
          <label>Price in rupees</label>
          <input name="price" type="number" min="1"
            max="10000000" required>
          <label>Size</label>
          <select name="size">
            <option>S</option><option selected>M</option>
            <option>L</option><option>XL</option>
          </select>
          <label>Condition</label>
          <select name="condition">
            <option>Excellent</option>
            <option>Very Good</option><option>Good</option>
          </select>
          <button type="submit">Add Listing</button>
        </form>
        <p id="sellMessage"></p>
      </div>
    </section>

    <section id="commission">
      <h2>ReWear Commission Calculator</h2>
      <div class="panel">
        <label>Clothing price (Rs.)</label>
        <input id="price" type="number" value="2000" min="0">
        <button onclick="calculate()">Calculate Earnings</button>
        <p id="result">ReWear: Rs. 240 · Seller: Rs. 1,760</p>
      </div>
    </section>

    <footer>ReWear © 2026 · Fashion with a second life ♻</footer>

    <script>
    async function buy(id) {
      const r = await fetch('/api/orders', {
        method: 'POST',
        headers: {'Content-Type':'application/json'},
        body: JSON.stringify({product_id:id})
      });
      const d = await r.json();
      alert(d.message || d.detail || 'Please try again');
    }

    function calculate() {
      const p = Number(document.getElementById('price').value);
      if (!Number.isFinite(p) || p < 0) {
        alert('Enter a valid price');
        return;
      }
      const fee = p * 0.12;
      document.getElementById('result').textContent =
        'ReWear commission: Rs. ' + fee.toFixed(2) +
        ' | Seller earns: Rs. ' + (p-fee).toFixed(2);
    }

    document.getElementById('sellForm').onsubmit =
      async function(e) {
        e.preventDefault();
        const f = new FormData(this);
        const data = Object.fromEntries(f.entries());
        data.price = Number(data.price);
        data.emoji = '👕';
        const r = await fetch('/api/products', {
          method:'POST',
          headers:{'Content-Type':'application/json'},
          body:JSON.stringify(data)
        });
        const d = await r.json();
        document.getElementById('sellMessage').textContent =
          d.message || d.detail || 'Done';
        if (r.ok) setTimeout(() => location.reload(), 700);
      };

    function filterProducts() {
      const q = document.getElementById('search').value.toLowerCase();
      document.querySelectorAll('#products .card').forEach(card => {
        card.style.display = card.innerText.toLowerCase().includes(q)
          ? '' : 'none';
      });
    }
    </script>
    </body>
    </html>
    """


@app.post("/api/products")
def add_product(p: NewProduct):
    if p.price <= 0:
        from fastapi import HTTPException
        raise HTTPException(400, "Price must be greater than zero")

    item = p.model_dump()
    item["id"] = max((x["id"] for x in products), default=0) + 1
    products.append(item)
    return {"message": "Listing added successfully!"}


@app.post("/api/orders")
def place_order(order: NewOrder):
    product = next(
        (p for p in products if p["id"] == order.product_id), None
    )
    if product is None:
        from fastapi import HTTPException
        raise HTTPException(404, "Product not found")

    price = product["price"]
    commission = round(price * commission_rate, 2)
    seller_earning = round(price - commission, 2)

    record = {
        "order_id": len(orders) + 1,
        "product": product["name"],
        "price": price,
        "commission": commission,
        "seller_earning": seller_earning,
        "status": "Demo order"
    }
    orders.append(record)

    return {
        "message": (
            f"Demo order created! Price Rs. {price:,.2f}; "
            f"ReWear commission Rs. {commission:,.2f}; "
            f"seller earning Rs. {seller_earning:,.2f}. "
            "No payment was collected."
        ),
        "order": record
    }


@app.get("/api/orders")
def get_orders():
    return orders


Overwriting main.py


In [3]:

import subprocess
import sys

server = subprocess.Popen([
    sys.executable, "-m", "uvicorn",
    "main:app", "--host", "0.0.0.0", "--port", "8000"
])
print("Server started. Open http://127.0.0.1:8000")


Server started. Open http://127.0.0.1:8000
